[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frhack/oli_ai/blob/main/notebooks/oli_ai_porte_logiche_didattica_attivazione.ipynb)

# Una rete impara la porta XOR

Nei due notebook precedenti (OR e NOR) abbiamo visto che **un neurone solo** è in grado di imparare le porte **linearmente separabili**: prima un neurone lineare per OR, poi un neurone affine (con bias) per NOR.

Ora proviamo con **XOR**, che invece **non è linearmente separabile** — non esiste una retta che separa i punti `(0,1)` e `(1,0)` (output 1) dai punti `(0,0)` e `(1,1)` (output 0). Un solo neurone affine, lo abbiamo già visto, non basta.

In questo notebook costruiamo una **rete di neuroni** (architettura `[2, 8, 1]`: 2 input, 8 nascosti, 1 output) e proviamo a farle imparare XOR. Vedremo che senza un ingrediente fondamentale **non basta nemmeno la rete**, e capiremo qual è l'ingrediente mancante.

## Setup

In [ ]:
!pip install --upgrade --no-cache-dir oli_ai > /dev/null

import jax
import jax.numpy as jnp
import jax.random as jr
from jax import grad
import numpy as np
import matplotlib.pyplot as plt
from oli_ai.mnist_lib import show_nn_graph
from oli_ai.porte_logiche import identity
from jax.numpy import tanh
from jax.nn import relu, sigmoid

## 1. I dati — la porta XOR

XOR (OR esclusivo): vale 1 quando **uno solo** dei due input è 1, vale 0 negli altri due casi (entrambi 0 o entrambi 1).

| x1 | x2 | XOR |
|:--:|:--:|:---:|
|  0 |  0 |  0  |
|  0 |  1 |  1  |
|  1 |  0 |  1  |
|  1 |  1 |  0  |

In [ ]:
X = jnp.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])
y = jnp.array([
    0.0,
    1.0,
    1.0,
    0.0,
])

## 2. L'architettura della rete

Per affrontare XOR servono **più neuroni**. Costruiamo una rete con:

- **2 neuroni di input** (un input per ogni cifra: x1, x2)
- **8 neuroni nascosti** (lo strato intermedio che fa il lavoro)
- **1 neurone di output** (il risultato 0 o 1)

Visualizziamo la struttura e contiamo i parametri.

In [ ]:
show_nn_graph([2, 8, 1])

# Quanti parametri ha la rete?
# - tra input e nascosti:  matrice W1 di forma (2, 8) -> 16 pesi
#                          + bias b1 di lunghezza 8   ->  8 bias
# - tra nascosti e output: matrice W2 di forma (8,)   ->  8 pesi
#                          + bias b2 (uno solo)       ->  1 bias
print('Parametri totali:', 2*8 + 8 + 8 + 1)

## 3. Il modello — `predict` con una variabile chiave: `attivazione`

Definiamo la rete come una funzione `predict(params, X)`. Il calcolo è in due passi:

1. **Primo strato**: ogni neurone nascosto fa una combinazione lineare degli input + bias, poi applica una funzione di **attivazione** — `h = attivazione(X · W1 + b1)`
2. **Secondo strato**: combinazione lineare degli 8 nascosti + bias finale — `out = h · W2 + b2`

⚠️ La variabile `attivazione` qui sotto è **il punto su cui torneremo nell'esercizio finale**: per ora la lasciamo a `identity`, cioè la funzione `f(x) = x` che restituisce il suo input invariato (in altre parole: **nessuna trasformazione** tra i due strati).

In [ ]:
attivazione = identity   # f(x) = x — nessuna non-linearità

def predict(params, X):
    W1, b1, W2, b2 = params
    h = attivazione(X @ W1 + b1)   # primo strato + attivazione
    return h @ W2 + b2             # secondo strato (lineare)

## 4. La loss

Stessa loss dei notebook precedenti — somma degli scarti quadratici tra predizione e target.

In [ ]:
def loss(params, X, y):
    y_hat = predict(params, X)
    return jnp.sum((y - y_hat) ** 2)

## 5. Inizializzazione dei parametri

⚠️ Una differenza importante rispetto ai notebook precedenti: con una rete a più strati **non possiamo partire con tutti i pesi a zero**. Se lo facessimo, tutti gli 8 neuroni nascosti farebbero la stessa cosa (tutti uguali, simmetricamente) e i gradienti rimarrebbero bloccati. Inizializziamo allora i pesi con piccoli numeri **casuali**.

Stampiamo le forme (`shape`) di ogni tensore di parametri così da renderci conto di cosa contiene la nostra rete.

In [ ]:
key = jr.PRNGKey(42)
k1, k2 = jr.split(key)

W1 = jr.normal(k1, (2, 8)) * 0.5   # 2 input -> 8 nascosti
b1 = jnp.zeros(8)
W2 = jr.normal(k2, (8,))   * 0.5   # 8 nascosti -> 1 output
b2 = jnp.array(0.0)

params = [W1, b1, W2, b2]

print(f'W1 shape: {W1.shape}    -> {W1.size} pesi')
print(f'b1 shape: {b1.shape}    -> {b1.size} bias')
print(f'W2 shape: {W2.shape}      -> {W2.size} pesi')
print(f'b2 shape: {b2.shape}        -> 1 bias')

print('L iniziale =', float(loss(params, X, y)))

## 6. Il gradiente e il loop di addestramento

Il `grad` di JAX calcola il gradiente per noi: gli passiamo la funzione `loss` e ci restituisce la funzione che, dati i parametri, calcola le derivate parziali.

Il loop è identico ai notebook precedenti — l'unica differenza è che `params` ora è una **lista di 4 tensori** invece di un singolo array, quindi aggiorniamo ogni elemento separatamente.

In [ ]:
grad_loss = grad(loss)

lr       = 0.1
n_epoche = 5000
storia_loss = []

for epoca in range(n_epoche):
    perdita = loss(params, X, y)
    g       = grad_loss(params, X, y)
    storia_loss.append(float(perdita))

    if epoca % 500 == 0:
        print(f'epoca {epoca:>5}: loss = {float(perdita):.4f}')

    # update: per ogni parametro, sottrai lr * il suo gradiente
    for i in range(len(params)):
        params[i] = params[i] - lr * g[i]

print(f'epoca {n_epoche:>5}: loss = {float(loss(params, X, y)):.4f}')

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(storia_loss, color='#38bdf8')
plt.xlabel('epoca'); plt.ylabel('loss'); plt.title('Andamento della loss durante il training')
plt.grid(alpha=0.3); plt.show()

## 7. Le predizioni della rete addestrata

Vediamo cosa risponde la rete sui 4 input di XOR.

In [ ]:
y_hat = predict(params, X)

print(f"{'x1':>3} {'x2':>3} {'target':>8} {'pred':>10} {'arrotondato':>12}")
print('-' * 40)
for (x_in, t, p) in zip(X, y, y_hat):
    print(f"{int(x_in[0]):>3} {int(x_in[1]):>3} {int(t):>8} {float(p):>10.4f} {int(round(float(p))):>12}")

**Le predizioni sono tutte ~0.5.** Il modello sta dicendo *"non lo so"* per ogni input. Eppure la rete ha 33 parametri (vs 3 del singolo neurone con bias) e abbiamo allenato per 5000 epoche! Cosa sta succedendo?

## 8. Perché la rete non riesce a imparare XOR?

Il problema è **la funzione `attivazione = identity`** che abbiamo usato.

Ricordiamoci cosa fa `identity`: restituisce il suo input invariato. Quindi il calcolo della rete diventa:

$$\text{out} = (X \cdot W_1 + b_1) \cdot W_2 + b_2$$

Espandendo:

$$\text{out} = X \cdot (W_1 \cdot W_2) + (b_1 \cdot W_2 + b_2)$$

Cioè: la nostra rete a **due strati** è matematicamente **equivalente** a un singolo neurone affine con peso $W' = W_1 \cdot W_2$ e bias $b' = b_1 \cdot W_2 + b_2$.

> **La composizione di funzioni affini è ancora una funzione affine.** Per quanti strati impili, finché tra di essi non c'è una *non-linearità*, la rete non è più potente di un singolo neurone affine — e quello, lo abbiamo già visto nel notebook NOR, non riesce a separare XOR.

Per spezzare la "linearità della catena" serve mettere tra uno strato e l'altro una **funzione non lineare**: la **funzione di attivazione**.

## 9. Esercizio — fai funzionare la rete

Per spezzare la simmetria affine che blocca il nostro modello ci serve una funzione di attivazione **non lineare**. La più semplice possibile si chiama **ReLU** ("Rectified Linear Unit"):

$$\text{relu}(x) \,=\, \max(0, x) \,=\, \begin{cases} x & \text{se } x > 0 \\ 0 & \text{se } x \le 0 \end{cases}$$

In parole: ReLU lascia invariati i numeri positivi e **rettifica a zero** quelli negativi. È letteralmente la più piccola modifica che si possa fare a `identity` per renderla non lineare. Vediamola graficamente.

In [ ]:
# Visualizzazione: la differenza tra identity (lineare) e relu (rettificata)
xs = jnp.linspace(-3, 3, 200)

plt.figure(figsize=(6, 4))
plt.plot(xs, identity(xs), '--', color='#94a3b8', label='identity (lineare)')
plt.plot(xs, relu(xs),     '-',  color='#0ea5e9', label='relu (rettificata)', linewidth=2)
plt.axhline(0, color='gray', lw=0.5); plt.axvline(0, color='gray', lw=0.5)
plt.xlabel('x'); plt.ylabel('f(x)')
plt.title('La rettificazione di ReLU')
plt.legend(); plt.grid(alpha=0.3); plt.show()

Il "gomito" a $x = 0$ è la chiave: **basta quel piegamento** per far uscire la nostra rete dal mondo affine. Ora fai la sostituzione.

Nella **Sez. 3** sostituisci

```python
attivazione = identity
```

con

```python
attivazione = relu
```

Dopo aver modificato la riga, riesegui:

1. la cella della Sez. 3 (per ridefinire `predict`)
2. la cella della Sez. 5 (per re-inizializzare i parametri)
3. la cella della Sez. 6 (loop di training)
4. la cella della Sez. 7 (predizioni)

Dovresti vedere la loss scendere molto più giù di prima e le predizioni diventare vicine ai target $\{0, 1, 1, 0\}$. La rete con la **stessa architettura** ma con la non-linearità giusta riesce ora a imparare XOR.

**Domande da pensare**:

- Cosa succede se al posto di `relu` provi `tanh` (la tangente iperbolica, curva a "S" che mappa in $(-1, 1)$) o `sigmoid` (anch'essa una curva a "S", ma che mappa in $(0, 1)$)? Le predizioni cambiano? La loss converge in modo più o meno regolare?
- E se torni a `identity` ma aumenti `n_epoche` a 50.000? Riesce a imparare? (Spoiler: no, e ora sai perché.)

## 10. Visualizzazione 3D — come la rete deforma lo spazio

Ora che la rete impara XOR (perché hai aggiunto la non-linearità ReLU), possiamo "guardare" cosa sta facendo davvero. La rete prende ogni input $(x_1, x_2)$ e gli associa un numero — l'output. Se disegniamo questo output come **altezza** sopra il piano $(x_1, x_2)$, otteniamo una **superficie 3D**.

- Nei 4 angoli $(0,0)$, $(0,1)$, $(1,0)$, $(1,1)$ la superficie passa per i valori target di XOR: rispettivamente 0, 1, 1, 0.
- Tra un angolo e l'altro, la rete sta usando i suoi 8 neuroni nascosti + ReLU per "piegare" lo spazio.

**Il piano grigio orizzontale a quota 0.5** rappresenta la soglia di decisione: tutto ciò che sta sopra → classe **1**, tutto sotto → classe **0**. Vedrai che il piano taglia la superficie esattamente "in diagonale", separando perfettamente i due punti rossi (classe 0) dai due verdi (classe 1) — cosa che con una rete lineare/affine (senza attivazione) sarebbe **impossibile**, perché la superficie sarebbe un piano e nessun altro piano potrebbe separare quattro punti disposti come XOR.

**Ruota il grafico** con il mouse per vederlo da angoli diversi: si vede chiaramente la "sella" che la rete ha imparato a costruire.

In [ ]:
import plotly.graph_objects as go

# Griglia 2D sul dominio degli input (con un po' di margine)
nx, ny = 60, 60
xs = jnp.linspace(-0.3, 1.3, nx)
ys = jnp.linspace(-0.3, 1.3, ny)
X_grid, Y_grid = jnp.meshgrid(xs, ys)
grid_pts = jnp.stack([X_grid.flatten(), Y_grid.flatten()], axis=1)

# Calcoliamo l'output della rete su ogni punto della griglia
Z = predict(params, grid_pts).reshape(X_grid.shape)

# I 4 punti XOR (input + output reale + label)
corners_x = [0.0, 0.0, 1.0, 1.0]
corners_y = [0.0, 1.0, 0.0, 1.0]
labels    = [0,   1,   1,   0  ]  # XOR
corners_z = [float(predict(params, jnp.array([[x, y]]))[0]) for x, y in zip(corners_x, corners_y)]

# Piano di decisione orizzontale a z = 0.5
plane_x = [-0.3, 1.3]
plane_y = [-0.3, 1.3]
plane_z = [[0.5, 0.5], [0.5, 0.5]]

import numpy as np
fig = go.Figure()
# superficie della rete
fig.add_trace(go.Surface(
    x=np.asarray(xs), y=np.asarray(ys), z=np.asarray(Z),
    opacity=0.85, colorscale='RdBu_r', showscale=False, name='rete(x1, x2)',
))
# piano di decisione semi-trasparente
fig.add_trace(go.Surface(
    x=plane_x, y=plane_y, z=plane_z,
    opacity=0.30, colorscale=[[0, '#888888'], [1, '#888888']], showscale=False,
    name='piano decisione (z=0.5)',
))
# i 4 punti XOR
fig.add_trace(go.Scatter3d(
    x=corners_x, y=corners_y, z=corners_z,
    mode='markers+text',
    text=[f'  ({int(x)},{int(y)}) → {l}' for x, y, l in zip(corners_x, corners_y, labels)],
    textposition='top center',
    marker=dict(
        size=10,
        color=['#ef4444' if l == 0 else '#10b981' for l in labels],
        line=dict(color='black', width=1),
    ),
    name='punti XOR',
    showlegend=False,
))
fig.update_layout(
    title='Superficie della rete sul piano (x1, x2). Verde = classe 1, Rosso = classe 0. Il piano grigio è la soglia di decisione.',
    scene=dict(
        xaxis=dict(title='x1', range=[-0.3, 1.3]),
        yaxis=dict(title='x2', range=[-0.3, 1.3]),
        zaxis=dict(title='output della rete'),
        camera=dict(eye=dict(x=1.6, y=1.6, z=1.0)),
    ),
    width=760, height=560,
    margin=dict(l=0, r=0, b=0, t=50),
)
fig.show()

## Riepilogo — i tre passi del percorso

| Notebook | Modello | Cosa impara | Cosa NON impara |
|---|---|---|---|
| `_didattica` | un neurone **lineare** $w_1 x_1 + w_2 x_2$ | OR (e AND) | NOR |
| `_didattica_bias` | un neurone **affine** $w_1 x_1 + w_2 x_2 + b$ | NOR | XOR |
| `_didattica_attivazione` (questo) | rete `[2, 8, 1]` con **attivazione non lineare** | XOR (e tutte le porte logiche) | — |

L'aggiunta del **bias** ha permesso di imparare problemi che si risolvono con una retta non passante per l'origine. L'aggiunta della **funzione di attivazione** ha permesso di imparare problemi che non si risolvono affatto con una retta. Sono due salti concettuali distinti, e questa è esattamente l'evoluzione storica delle reti neurali.